#### EDA 04: Payments table
**File:** `olist_order_payments_dataset.csv`  |  **Goal:** understand how orders are paid, using this table only.

#### Load and first look
**Question:** how big is the table and what does a row look like?

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

RAW = Path("../../data/raw")                      # path from the notebook folder to the raw data
payments_raw = pd.read_csv(RAW / "olist_order_payments_dataset.csv")
payments = payments_raw.copy()                    # all changes happen on the copy, raw stays untouched

print("shape:", payments.shape)
print(payments.dtypes)
payments.head(3)

shape: (103886, 5)
order_id                    str
payment_sequential        int64
payment_type                str
payment_installments      int64
payment_value           float64
dtype: object


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71


- 103,886 rows and 5 columns: order, payment number, payment type, number of installments and payment value.
- `payment_sequential` is the number of the payment inside an order (1, 2, 3...). One row is one payment line, not one order.
- Types are fine: numbers are numeric, `payment_type` is text.

#### Keys and duplicates
**Question:** what identifies a row, and is any row repeated?

In [3]:
print("fully duplicated rows                  :", payments.duplicated().sum())
print("repeated (order_id, payment_sequential):", payments.duplicated(["order_id", "payment_sequential"]).sum())

fully duplicated rows                  : 0
repeated (order_id, payment_sequential): 0


- No duplicated rows. The pair (`order_id`, `payment_sequential`) is unique, so it is the key of this table.
- `order_id` alone is not a key. 

#### Missing values
**Question:** which columns have gaps?

In [3]:
# Column-wise missing count and percentage, for every column (zeros included)
missing = pd.DataFrame({"n_rows": len(payments),
                        "n_missing": payments.isna().sum(),
                        "pct_missing": (payments.isna().mean() * 100).round(2)})
display(missing.sort_values("n_missing", ascending=False))

,n_rows,n_missing,pct_missing
order_id,103886,0,0.0
payment_sequential,103886,0,0.0
payment_type,103886,0,0.0
payment_installments,103886,0,0.0
payment_value,103886,0,0.0


- No column has a missing value.

#### Payment lines per order
**Question:** how many payment lines does an order have?

In [4]:
lines_per_order = payments.groupby("order_id").size()

# Group 6 and more together for display
bucket = lines_per_order.clip(upper=6)
dist = bucket.value_counts().sort_index().rename("n_orders").to_frame()
dist.index = [str(i) if i < 6 else "6+" for i in dist.index]
dist.index.name = "payment_lines_per_order"
dist["pct_orders"] = (dist["n_orders"] / len(lines_per_order) * 100).round(2)
display(dist)

multi = lines_per_order[lines_per_order > 1]
print("distinct orders with payments        :", len(lines_per_order))
print("orders with more than 1 payment line :", len(multi), f"({len(multi) / len(lines_per_order) * 100:.1f}%)")
print("payment lines inside those orders    :", multi.sum(), f"({multi.sum() / len(payments) * 100:.1f}% of all lines)")
print("maximum lines in one order           :", lines_per_order.max())

,n_orders,pct_orders
payment_lines_per_order,,
1,96479,97.02
2,2382,2.40
3,301,0.30
4,108,0.11
5,52,0.05
6+,118,0.12


distinct orders with payments        : 99440
orders with more than 1 payment line : 2961 (3.0%)
payment lines inside those orders    : 7407 (7.1% of all lines)
maximum lines in one order           : 29


- 99,440 orders have payment lines. The orders table has 99,441 rows (raw data summary), so one order may have no payment. This is checked after the joins.
- 96,479 orders (97.0%) have a single payment line.
- 2,961 orders (3.0%) have more than one line, and they hold 7,407 lines (7.1% of all lines).
- The largest order has 29 payment lines. Its first lines are small vouchers (shown at the end of the outlier section).

#### Payment type
**Question:** how do customers pay, and do the types behave differently?

In [6]:
type_tbl = payments.groupby("payment_type").agg(n_lines=("order_id", "size"),
                                                median_value=("payment_value", "median"),
                                                mean_value=("payment_value", "mean"),
                                                max_value=("payment_value", "max"))
type_tbl["pct_lines"] = (type_tbl["n_lines"] / len(payments) * 100).round(2)
display(type_tbl.sort_values("n_lines", ascending=False).round(2))

print("installments by type")
payments.groupby("payment_type")["payment_installments"].agg(["min", "median", "max"])

,n_lines,median_value,mean_value,max_value,pct_lines
payment_type,,,,,
credit_card,76795,106.87,163.32,13664.08,73.92
boleto,19784,93.89,145.03,7274.88,19.04
voucher,5775,39.28,65.70,3184.34,5.56
debit_card,1529,89.30,142.57,4445.50,1.47
not_defined,3,0.00,0.00,0.00,0.00


installments by type


,min,median,max
payment_type,,,
boleto,1,1.0,1
credit_card,0,3.0,24
debit_card,1,1.0,1
not_defined,1,1.0,1
voucher,1,1.0,1


- Credit card is 73.9% of lines (76,795), boleto 19.0%, voucher 5.6% and debit card 1.5%.
- Only 3 lines have the type `not_defined`. All three have a value of 0 (see the value section). It is an unknown type, not a real fifth payment method.
- Vouchers are small: median value 39.28, against 106.87 for credit card. They look like partial payments.
- Installments above 1 exist only for credit card. Boleto, debit card and voucher always have 1.

#### Installments
**Question:** how are installments spread, and are there invalid values?

In [7]:
inst_bins = pd.cut(payments["payment_installments"], [-1, 0, 1, 3, 6, 9, 10, 100],
                   labels=["0", "1", "2-3", "4-6", "7-9", "10", "11+"])
inst = inst_bins.value_counts().sort_index().rename("n_lines").to_frame()
inst["pct"] = (inst["n_lines"] / len(payments) * 100).round(2)
display(inst)

print("lines with 0 installments:")
display(payments[payments["payment_installments"] == 0])

cc = payments[payments["payment_type"] == "credit_card"]
print("credit card lines                  :", len(cc))
print("credit card with more than 1 inst. :", f"{(cc['payment_installments'] > 1).mean() * 100:.1f}%")
print("credit card median installments    :", cc["payment_installments"].median())
print("credit card with 10 or more        :", (cc["payment_installments"] >= 10).sum())

# Installments grow with the payment value?
print("\ncredit card installments by payment value")
by_value = pd.cut(cc["payment_value"], [0, 50, 100, 200, 500, 1e9], labels=["<50", "50-100", "100-200", "200-500", "500+"])
cc.groupby(by_value, observed=True)["payment_installments"].agg(n_lines="count", median="median", mean="mean").round(2)

,n_lines,pct
payment_installments,,
0,2,0.00
1,52546,50.58
2-3,22874,22.02
4-6,16257,15.65
7-9,6538,6.29
10,5328,5.13
11+,341,0.33


lines with 0 installments:


,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


credit card lines                  : 76795
credit card with more than 1 inst. : 66.9%
credit card median installments    : 3.0
credit card with 10 or more        : 5669

credit card installments by payment value


,n_lines,median,mean
payment_value,,,
<50,13380,1.0,1.69
50-100,22529,2.0,2.73
100-200,24642,3.0,3.82
200-500,12756,5.0,5.18
500+,3488,8.0,7.18


- 52,546 lines (50.6%) have a single installment. The maximum is 24.
- 2 lines have 0 installments. Both are credit card payments (payment number 2), which is not possible for a card, so the value is invalid.
- 66.9% of credit card lines use more than one installment (median 3). 5,669 lines use 10 or more.
- Installments rise clearly with the amount: the median is 1 below 50, 3 between 100 and 200, and 8 above 500.
- **Decision:** keep the 2 zero values and treat them as unknown in the pipeline. Do not replace them with 1 inside the data.

#### Mixed payments
**Question:** do orders combine payment types?

In [9]:
combo = payments.groupby("order_id")["payment_type"].agg(lambda s: "+".join(sorted(s.unique())))
display(combo.value_counts().rename("n_orders").to_frame())

print("orders using more than one type:", combo.str.contains(r"\+").sum())
print("orders with any voucher        :", combo.str.contains("voucher").sum())
print("orders paid by voucher only    :", (combo == "voucher").sum())

,n_orders
payment_type,
credit_card,74259
boleto,19784
credit_card+voucher,2245
voucher,1621
debit_card,1527
not_defined,3
credit_card+debit_card,1


orders using more than one type: 2246
orders with any voucher        : 3866
orders paid by voucher only    : 1621


- Most orders use one type: credit card only (74,259) or boleto only (19,784).
- 2,246 orders mix types, and 2,245 of them are credit card plus voucher. Only 1 order is credit card plus debit card.
- 3,866 orders use a voucher, and 1,621 are paid by voucher only.
- A voucher is therefore often a part payment on top of a card, which explains the many small lines.

#### Outliers
**Question:** are the extreme payment amounts and installments real?

In [10]:
print("lines with value > 1000:", (payments["payment_value"] > 1000).sum())
print("lines with value > 5000:", (payments["payment_value"] > 5000).sum())

print("\nlargest payment lines")
display(payments.nlargest(3, "payment_value"))

print("the order with the most payment lines (first 5 lines)")
top_order = lines_per_order.idxmax()
payments[payments["order_id"] == top_order].sort_values("payment_sequential").head(5)

lines with value > 1000: 1150
lines with value > 5000: 6

largest payment lines


,order_id,payment_sequential,payment_type,payment_installments,payment_value
52107,03caa2c082116e1d31e67e9ae3700499,1,credit_card,1,13664.08
34370,736e1922ae60d0d6a89247b851902527,1,boleto,1,7274.88
41419,0812eb902a67711a1cb742b3cdaa65ae,1,credit_card,8,6929.31


the order with the most payment lines (first 5 lines)


,order_id,payment_sequential,payment_type,payment_installments,payment_value
14321,fa65dad1b0e818e3ccc5cb0e39231352,1,voucher,1,3.71
23074,fa65dad1b0e818e3ccc5cb0e39231352,2,voucher,1,8.51
65641,fa65dad1b0e818e3ccc5cb0e39231352,3,voucher,1,2.95
9985,fa65dad1b0e818e3ccc5cb0e39231352,4,voucher,1,29.16
28330,fa65dad1b0e818e3ccc5cb0e39231352,5,voucher,1,0.66


- 1,150 lines are above 1,000 and 6 are above 5,000. The largest is 13,664.08 paid by credit card in 1 installment.
- The next ones (7,274.88 boleto, 6,929.31 credit card in 8 installments) are plausible high-value purchases.
- The order with 29 payment lines is made of small voucher payments (the first five are 3.71, 8.51, 2.95, 29.16 and 0.66).